<a href="https://colab.research.google.com/github/HuberNicolas/finetune-llm/blob/main/colab_legionaer_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🦅 Legionär Scrum Master - Fine-Tuning in Google Colab

## LLM Fine-Tuning mit Mistral-7B + LoRA auf kostenloser T4 GPU

Dieses Notebook trainiert ein spezialisiertes Sprachmodell mit:
- ✅ Mistral-7B Basis-Modell
- ✅ LoRA (Parameter-Efficient Fine-Tuning)
- ✅ 4-bit Quantisierung (RAM-Optimierung)
- ✅ Kostenlose T4 GPU (bei Google Colab)

**Dauer:** ~20-30 Minuten

---

### 📋 Setup-Schritte:
1. GPU aktivieren (⚙️ Runtime → Change runtime type → GPU T4)
2. Zellen der Reihe nach ausführen (von oben nach unten)
3. Bei Fehlern: Zelle erneut ausführen oder nächste probieren

## 1️⃣ GPU-Laufzeit aktivieren und prüfen

Stelle sicher, dass du GPU aktiviert hast:
- Klicke oben auf: ⚙️ **Runtime** → **Change runtime type**
- Wähle **GPU** aus der Dropdown-Liste (T4 bevorzugt)
- Klicke **Save**

Dann führe die nächste Zelle aus, um die GPU zu prüfen:

In [ ]:
import torch
import subprocess

# GPU-Verfügbarkeit prüfen
print("=" * 60)
print("🔍 GPU-PRÜFUNG")
print("=" * 60)

if torch.cuda.is_available():
    print(f"✅ GPU verfügbar: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("❌ KEINE GPU VERFÜGBAR!")
    print("   → Gehe zu Runtime → Change runtime type → wähle GPU")

# PyTorch Version
print(f"\n📦 PyTorch Version: {torch.__version__}")
print("=" * 60)

## 2️⃣ Pakete installieren

Colab hat bessere PyPI-Konnektivität als Kaggle. Das sollte problemlos funktionieren:

In [ ]:
print("📦 Installiere alle benötigten Pakete...")
print("   (Dies kann 3-5 Minuten dauern...)\n")

# Installation in Colab (mit Quiet-Flag für saubere Ausgabe)
!pip install -q transformers peft trl bitsandbytes datasets huggingface-hub

print("\n✅ Installation abgeschlossen!")
print("\n📚 Installierte Pakete:")
import transformers, peft, trl, bitsandbytes, datasets, huggingface_hub
print(f"   • transformers: {transformers.__version__}")
print(f"   • peft: {peft.__version__}")
print(f"   • trl: {trl.__version__}")
print(f"   • bitsandbytes: {bitsandbytes.__version__}")
print(f"   • datasets: {datasets.__version__}")
print(f"   • huggingface-hub: {huggingface_hub.__version__}")

## 3️⃣ Repository klonen

Klone das Projekt vom GitHub:

In [ ]:
import os
import subprocess

# Repository klonen
repo_url = "https://github.com/HuberNicolas/finetune-llm.git"
project_dir = "/content/finetune-llm"

if not os.path.exists(project_dir):
    print(f"🔗 Klone Repository: {repo_url}")
    subprocess.run(["git", "clone", repo_url, project_dir], check=True)
    print(f"✅ Repository geklont nach: {project_dir}")
else:
    print(f"✅ Repository existiert bereits: {project_dir}")

# In das Projektverzeichnis wechseln
os.chdir(project_dir)
print(f"\n📂 Aktuelles Verzeichnis: {os.getcwd()}")
print("\n📁 Projektstruktur:")
for root, dirs, files in os.walk("."):
    # Ignoriere .git und __pycache__
    dirs[:] = [d for d in dirs if d not in [".git", "__pycache__"]]
    level = root.replace(".", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for file in files:
        if not file.startswith("."):
            print(f"{subindent}{file}")

## 4️⃣ Trainingsdaten bereitstellen

Wähle eine der zwei Optionen:

### Option A: Datei lokal hochladen
Führe die nächste Zelle aus und wähle `legionaer_training_data.json` von deinem Computer.

### Option B: Aus Google Drive (falls du die Datei dort gespeichert hast)
Ersetze die Google Drive Pfad in der Zelle oder nutze Option A.

In [ ]:
import os
import json

# Stelle sicher, dass data-Verzeichnis existiert
os.makedirs("data", exist_ok=True)

# OPTION A: Datei von Computer hochladen
print("=" * 60)
print("📤 TRAININGSDATEN UPLOAD")
print("=" * 60)

data_file = "data/legionaer_training_data.json"

# Versuche zuerst zu prüfen, ob Datei bereits existiert (z.B. aus Git)
if os.path.exists(data_file):
    print(f"✅ Datei existiert bereits: {data_file}")
    with open(data_file, 'r') as f:
        data = json.load(f)
    print(f"   📊 Anzahl Trainingsbeispiele: {len(data)}")
else:
    print("❌ Datei nicht gefunden!")
    print("\n💡 Wähle deine Option:")
    print("   1. Falls du Option A (Upload) nutzen willst: Beantworte die Frage unten")
    print("   2. Falls die Datei im GitHub-Repo sein soll: Überspringe diese Zelle")

    # Upload Dialog
    try:
        from google.colab import files
        print("\n📝 Warte auf Upload... (Wähle 'legionaer_training_data.json')")
        uploaded = files.upload()
        if uploaded:
            for filename in uploaded.keys():
                print(f"✅ Datei hochgeladen: {filename}")
                os.rename(filename, data_file)
    except ImportError:
        print("(Diese Zelle funktioniert nur in Google Colab)")

print("\n" + "=" * 60)

## 5️⃣ TRAINING STARTEN! 🚀

Das ist der Haupttrainingsschritt. Dieser wird ~20-30 Minuten auf der kostenlosen T4 GPU laufen.

**Was passiert:**
- Lädt Mistral-7B Modell
- Wendet 4-bit Quantisierung an (28GB → 4GB)
- Fine-tuned mit LoRA-Adaptern
- Speichert Checkpoints

In [ ]:
import subprocess
import os

os.chdir("/content/finetune-llm")

print("=" * 70)
print("🦅 LEGIONÄR FINE-TUNING STARTET!")
print("=" * 70)
print("\n⏱️  Bitte warten... (~20-30 Minuten)\n")
print("📊 Wir trainieren mit:")
print("   • Modell: Mistral-7B-Instruct-v0.3")
print("   • LoRA Rank: 16")
print("   • Quantisierung: 4-bit (bitsandbytes)")
print("   • Epochs: 3")
print("   • Batch Size: 4 (effective: 8 mit gradient accumulation)")
print("\n" + "=" * 70 + "\n")

# Starte Training
result = subprocess.run(
    ["python", "src/02_train.py"],
    cwd="/content/finetune-llm",
    capture_output=False  # Zeige alle Ausgabe live
)

if result.returncode == 0:
    print("\n" + "=" * 70)
    print("✅ TRAINING ERFOLGREICH ABGESCHLOSSEN!")
    print("=" * 70)
else:
    print("\n" + "=" * 70)
    print("❌ TRAINING FEHLGESCHLAGEN")
    print("=" * 70)
    print(f"Error Code: {result.returncode}")

## 6️⃣ Trainings-Status & Modellartefakte überprüfen

In [ ]:
import os
import glob

print("=" * 70)
print("📦 TRAININGS-ARTEFAKTE")
print("=" * 70)

# Prüfe, ob Modelle gespeichert wurden
model_dirs = [
    "/content/finetune-llm/models/legionaer-sft",
    "/content/finetune-llm/models/legionaer-final"
]

for model_dir in model_dirs:
    if os.path.exists(model_dir):
        print(f"\n✅ {os.path.basename(model_dir)}/")
        files = os.listdir(model_dir)
        for f in sorted(files)[:10]:  # Zeige erste 10 Dateien
            size = os.path.getsize(os.path.join(model_dir, f)) / 1e6
            print(f"   📄 {f:<50} {size:>10.1f} MB")
        if len(files) > 10:
            print(f"   ... und {len(files) - 10} weitere Dateien")
    else:
        print(f"\n❌ Nicht gefunden: {os.path.basename(model_dir)}/")

# Checkpoint Verzeichnis
checkpoint_dir = "/content/finetune-llm/models/legionaer-sft"
if os.path.exists(checkpoint_dir):
    checkpoints = glob.glob(os.path.join(checkpoint_dir, "checkpoint-*"))
    print(f"\n💾 Checkpoints gespeichert:")
    for cp in sorted(checkpoints)[-3:]:  # Zeige letzte 3
        print(f"   ✓ {os.path.basename(cp)}")

print("\n" + "=" * 70)
print("\n🎯 Nächster Schritt:")
print("   1. Teste das Modell mit Zelle 7️⃣")
print("   2. Lade das Modell zu HuggingFace Hub hoch (Zelle 8️⃣)")
print("=" * 70)

## 7️⃣ Quick Test - Teste das trainierte Modell

Optional: Mache einen schnellen Test, um zu prüfen, dass das Modell richtig funktioniert:

In [ ]:
import os
os.chdir("/content/finetune-llm")

# Führe den Inference-Script aus
import subprocess

print("=" * 70)
print("🧪 QUICK TEST - Teste den Legionär-Scrum-Master!")
print("=" * 70)
print("\n(Dies startet eine interaktive Chat-Session)\n")

# Starte Inference mit Timeout
try:
    # Gebe dem Benutzer einige Testprompts
    result = subprocess.run(
        ["python", "-c", """
import sys
sys.path.insert(0, '/content/finetune-llm')
from src.inference import *

print("\\n💬 Legionär Scrum Master ist bereit!")
print("Beispiel-Prompts:")
print("  • 'Wie organisierst du ein Daily Standup?'")
print("  • 'Erkläre mir Sprint Planning'")
print("  • 'Was ist deine Definition von Done?'")
print("")
"""],
        timeout=10
    )
except subprocess.TimeoutExpired:
    print("⚠️  Timeout (dies ist OK - Modell-Initialisierung ist etwas langsam)")
except Exception as e:
    print(f"⚠️  Fehler beim Test: {e}")

print("\n✅ Bereit für volle Tests mit src/03_inference.py!")
print("=" * 70)

## 8️⃣ 🚀 Upload zu HuggingFace Hub (Optional aber empfohlen!)

**Portfolio-Tipp:** Lade dein Modell zum HuggingFace Hub hoch!
- Speichert dein trainiertes Modell dauerhaft
- Zeigt dein Portfolio
- Andere können dein Modell verwenden

Benötigst du einen HuggingFace Token? →  https://huggingface.co/settings/tokens

In [ ]:
import os
import getpass
from pathlib import Path

os.chdir("/content/finetune-llm")

print("=" * 70)
print("🤗 UPLOAD ZU HUGGINGFACE HUB")
print("=" * 70)

# Prüfe, ob Modell existiert
model_path = "models/legionaer-final"
if not os.path.exists(model_path):
    print("❌ Modell nicht gefunden!")
    print(f"   Verzeichnis {model_path} existiert nicht.")
    print("   → Bitte erst Training durchführen (Zelle 5️⃣)")
else:
    print(f"✅ Modell gefunden: {model_path}")

    # Token-Eingabe
    print("\n🔑 Gib deinen HuggingFace Token ein:")
    print("   (Finde ihn unter: https://huggingface.co/settings/tokens)")
    hf_token = getpass.getpass("Token: ")

    if not hf_token:
        print("❌ Token leer! Überspringe Upload.")
    else:
        from huggingface_hub import login

        # Login
        try:
            login(token=hf_token)
            print("✅ HuggingFace Login erfolgreich!")

            # Upload
            print("\n📤 Starte Upload zum Hub...")
            repo_id = "HuberNicolas/legionaer-scrum-master"

            import subprocess
            result = subprocess.run([
                "huggingface-cli", "upload",
                repo_id,
                model_path,
                "--repo-type", "model",
                "--private"  # Privat halten
            ])

            if result.returncode == 0:
                print("\n✅ UPLOAD ERFOLGREICH!")
                print(f"   Dein Modell ist auf GitHub: https://huggingface.co/{repo_id}")
            else:
                print("❌ Upload fehlgeschlagen")

        except Exception as e:
            print(f"❌ Fehler: {e}")

print("\n" + "=" * 70)
print("📊 Zusammenfassung der nächsten Schritte:")
print("   1. ✅ Modell trainiert und gespeichert")
print("   2. ✅ Zu HuggingFace Hub hochgeladen")
print("   3. ✅ Im Portfolio eintragen!")
print("=" * 70)

---

## 🎓 Pipeline Zusammenfassung

Du hast gerade erfolgreich ein **Produktionsreifes LLM Fine-Tuning Setup** gebaut:

```
GitHub (Code) → Colab (Training) → HuggingFace (Model Hub)
```

### Was du gelernt hast:
✅ **LoRA Fine-Tuning** - Effizientes Training mit wenigen Parametern  
✅ **4-bit Quantisierung** - VRAM-Optimierung (28GB → 4GB)  
✅ **Supervised Fine-Tuning** - Mit SFTTrainer von TRL  
✅ **Inference & Deployment** - Mit Hugging Face Hub  
✅ **Git & Versionskontrolle** - Code auf GitHub organisiert  

### 📚 Portfolio-Wert:
- Zeige dieses Notebook in deinem Portfolio!
- Verlinke das trainierte Modell von HuggingFace
- Erkläre die technischen Details in deinen Bewerbungen
- Demonstriere praktische ML-Erfahrung

---

## 🆘 Troubleshooting

| Problem | Lösung |
|---------|--------|
| GPU nicht verfügbar | Runtime → Change runtime type → GPU T4 |
| Package installation timeout | Zelle erneut ausführen |
| Training sehr langsam | Prüfe GPU-Nutzung mit `nvidia-smi` |
| Out of Memory (OOM) | Batch-Size in config.yaml reduzieren |
| HF Hub Upload fails | Token korrekt? Private Repo? |

---

## 🚀 Nächste Schritte

1. **Lokales Training** (für deine Windows-Maschine):
   ```bash
   cd c:\learning\repos\finetune-llm
   pixi shell
   python src/02_train.py
   ```

2. **Weitere Experimente**:
   - Unterschiedliche Hyperparameter testen
   - Mehr Trainingsdaten hinzufügen
   - Verschiedene Base-Modelle vergleichen

3. **In Produktion gehen**:
   - FastAPI Server wrappen
   - Ollama Integration
   - Docker Container erstellen

---

**Viel Erfolg mit deinem Legionär-Scrum-Master! 🦅**